In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

# ================== 1. 读矩阵 + 三种初始化方式 ==================
# 读原始共现矩阵（基于训练集统计的 6×16）
df = pd.read_excel("D:/yrq/excel/gid/matrix_16x6.xlsx", index_col=0)
matrix_base = df.values.astype(np.float32)   # 形状：(6,16)

if matrix_base.shape != (6, 16):
    raise ValueError(f"matrix_16x6.xlsx 形状是 {matrix_base.shape}，预期应为 (6,16)")

# 选择初始化方式： 'cooc' / 'ones' / 'zeros'
INIT_MODE = "cooc"   # <<< 这里改成 'ones' 或 'zeros' 就是全 1 / 全 0

if INIT_MODE == "cooc":
    matrix_6x16 = matrix_base
elif INIT_MODE == "ones":
    matrix_6x16 = np.ones_like(matrix_base, dtype=np.float32)
elif INIT_MODE == "zeros":
    matrix_6x16 = np.zeros_like(matrix_base, dtype=np.float32)
else:
    raise ValueError(f"未知的 INIT_MODE: {INIT_MODE}")

print(f"[HI Loss] 使用的矩阵初始化方式: {INIT_MODE}, matrix_6x16.shape = {matrix_6x16.shape}")


# ================== 2. HI Loss ==================
class HILoss():
    def __init__(self, matrix_6x16):
        """
        matrix_6x16: np.array, 形状 [6,16]
        行：6 个粗类别
        列：16 个细类别

        本实现用：
        - 每一列的和 -> 细粒度类别的权重 w16
        - 每一行的和 -> 粗粒度类别的权重 w6
        再做一次归一化，让权重大致在 0~1 之间。
        """
        m = tf.constant(matrix_6x16, dtype=tf.float32)   # [6,16]

        # 按列求和 -> 每个细粒度类别（16 类）的总关联度
        col_sums = tf.reduce_sum(m, axis=0)              # [16]
        # 按行求和 -> 每个粗粒度类别（6 类）的总关联度
        row_sums = tf.reduce_sum(m, axis=1)              # [6]

        # 做归一化，避免数值过大
        max_col = tf.reduce_max(col_sums)
        max_row = tf.reduce_max(row_sums)

        # 注意：若矩阵全 0，max_col/max_row 为 0，此时直接设为 1（等价于纯 CE）
        self.w16 = tf.where(
            max_col > 0.0,
            col_sums / (max_col + 1e-8),
            tf.ones_like(col_sums)
        )  # [16]

        self.w6 = tf.where(
            max_row > 0.0,
            row_sums / (max_row + 1e-8),
            tf.ones_like(row_sums)
        )  # [6]

        # 打印一下，方便 sanity check
        print("[HI Loss] w16:", self.w16.numpy())
        print("[HI Loss] w6 :", self.w6.numpy())

    def loss_for_output16(self, y_true1, y_pred1):
        """
        16 类输出的 HI 损失：
        L = CE + (1 - w16[class])

        - y_true1: [B,H,W] 或 [B,H,W,1]
        - y_pred1: [B,H,W,16]
        """
        # 基础 sparse CE
        base_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true1, y_pred1)
        # [B,H,W]
        y_true1_int = tf.cast(y_true1, tf.int32)

        # 按类别索引取出对应 w16，形状与 y_true1 相同
        w = tf.gather(self.w16, y_true1_int)   # [B,H,W] 或 [B,H,W,1]

        adjusted_loss = base_loss + (1.0 - w)

        return tf.reduce_mean(adjusted_loss)

    def loss_for_output6(self, y_true2, y_pred2):
        """
        6 类输出的 HI 损失：
        L = CE + (1 - w6[class])
        """
        base_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true2, y_pred2)
        y_true2_int = tf.cast(y_true2, tf.int32)

        w = tf.gather(self.w6, y_true2_int)    # [B,H,W]

        adjusted_loss = base_loss + (1.0 - w)

        return tf.reduce_mean(adjusted_loss)


hi_loss = HILoss(matrix_6x16)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00045),
    loss={
        'out_16': hi_loss.loss_for_output16,
        'out_6' : hi_loss.loss_for_output6
    },
    metrics={
        'out_16': 'accuracy',
        'out_6' : 'accuracy'
    }
)

from tensorflow.keras.callbacks import LambdaCallback

train_losses, val_losses, train_acc, val_acc = [], [], [], []

def custom_on_epoch_end(epoch, logs):
    train_losses.append(logs['loss'])
    val_losses.append(logs.get('val_loss'))
    train_acc.append(logs['out_16_accuracy'])
    val_acc.append(logs.get('val_out_16_accuracy'))

hi_callback = LambdaCallback(on_epoch_end=custom_on_epoch_end)
